In [25]:
from sklearn.datasets import load_iris
import pandas as pd
import numpy as np

# Load the dataset
iris = load_iris()
X = iris.data.copy()
y = iris.target

# Convert to DataFrame
df = pd.DataFrame(X, columns=iris.feature_names)
df['Target'] = pd.Series(y).map(dict(enumerate(iris.target_names)))

# Introduce missing values randomly in input features (e.g., 10 values)
np.random.seed(42)  # for reproducibility
missing_rate = 0.05  # 5% missing

# Choose random indices to set as NaN
n_missing = int(np.floor(missing_rate * X.size))
missing_indices = [
    (np.random.randint(0, X.shape[0]), np.random.randint(0, X.shape[1]))
    for _ in range(n_missing)
]

for row, col in missing_indices:
    df.iat[row, col] = np.nan



In [41]:
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split , cross_val_score
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from mlxtend.feature_selection import ExhaustiveFeatureSelector
from sklearn.feature_selection import chi2 , SelectKBest
from sklearn.impute import SimpleImputer , MissingIndicator

In [30]:
# train test split
x_train, x_test , y_train , y_test = train_test_split(df.iloc[:,:-1] , df.iloc[:,-1] , random_state = 13 , test_size = 13)

In [33]:
x_train.isnull().sum()

,0
sepal length (cm),4
sepal width (cm),7
petal length (cm),6
petal width (cm),9


In [35]:

# for missing: 1 part of ml pipeline
# better to pass index then columns names in pieplines as numpy array is o/p which have index not names
# if no name and name is given code can cause error
tf1 = ColumnTransformer(
    transformers=[
        ("impute_sepal_length", SimpleImputer(strategy='mean'), [0]),
        ("impute_sepal_width", SimpleImputer(add_indicator=True), [1]),
        ("impute_petal_length", SimpleImputer(strategy='mean'), [2]),
        ("impute_petal_width", SimpleImputer(strategy='mean'), [3])
    ],
    remainder="passthrough"  # ✅ correctly placed outside of the list
)


In [40]:

tf2 = ColumnTransformer(
    transformers=[
        (
            "Hot_encode_target",                  # Name of the transformer
            OneHotEncoder(sparse_output=False,          # Return dense array
                          handle_unknown='ignore'),  # Avoid crashing on unseen labels
            [0]                                   # Apply to column at index 0
        )
    ]
)
# if want from col a : b ,use slice(a,b) in transfomer in index pos

In [42]:
# Feature sel
tf3 = SelectKBest(score_func=chi2 , k = 3)


In [43]:
#tf4 : model train
tf4 = LogisticRegression()

In [46]:
from sklearn.pipeline import Pipeline , make_pipeline


In [47]:

pipe = Pipeline([
    ("tf1", tf1),
    ("tf2", tf2),
    ("tf3", tf3),
    ("tf4", tf4)
])


In [48]:
pipe.fit(x_train , y_train)
# if u create a pipeline for preprocessing u will then use fit_trasnform()

Pipeline(steps=[('tf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_sepal_length',
                                                  SimpleImputer(), [0]),
                                                 ('impute_sepal_width',
                                                  SimpleImputer(add_indicator=True),
                                                  [1]),
                                                 ('impute_petal_length',
                                                  SimpleImputer(), [2]),
                                                 ('impute_petal_width',
                                                  SimpleImputer(), [3])])),
                ('tf2',
                 ColumnTransformer(transformers=[('Hot_encode_target',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [0])])),
                ('tf3',
                 SelectKBest(k=3,
                             score_func=<function chi2 at 0x7a39e7f04680>)),
                ('tf4', LogisticRegression())])

In [66]:
pipe.classes_
pipe.named_steps # complete steps dict
obj = pipe.named_steps['tf1'].transformers[0][1]


In [69]:
y_pred = pipe.predict(x_test)

In [70]:
accuracy_score(y_test , y_pred)

0.46153846153846156

In [1]:
import numpy as np
import pandas as pd

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic regression data
X, y = make_regression(
    n_samples=1000,        # good amount of data
    n_features=10,         # 10 numeric features
    n_informative=8,       # 8 features are useful
    noise=15.0,            # add noise to make it realistic
    random_state=42
)

# Convert to DataFrame
df = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(X.shape[1])])
df['target'] = y

# Introduce random missing values (approx 10% missing per column)
for col in df.columns[:-1]:  # exclude target
    missing_indices = np.random.choice(df.index, size=int(0.1 * len(df)), replace=False)
    df.loc[missing_indices, col] = np.nan

# Show sample

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import chi2 , VarianceThreshold ,f_classif , SelectKBest
from sklearn.preprocessing import MinMaxScaler , StandardScaler , RobustScaler
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator

from mlxtend.feature_selection import ExhaustiveFeatureSelector


In [60]:
df.isnull().sum()

,0
feature_0,100
feature_1,100
feature_2,100
feature_3,100
feature_4,100
feature_5,100
feature_6,100
feature_7,100
feature_8,100
feature_9,100


In [14]:
x_train , x_test , y_train , y_test = train_test_split(df.iloc[:,:-1] , df.iloc[:,-1] , random_state = 13, test_size = 12)


In [42]:
df

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,target
0,-0.707669,2.122156,-1.260884,0.917862,0.774634,-1.519370,1.266911,1.032465,-0.484234,0.443819,-93.571038
1,0.668655,-0.730956,-0.535335,0.358454,1.295872,0.685508,NaN,0.757922,1.848609,0.098068,105.964504
2,0.691619,-0.624510,-1.045529,-0.026261,-0.998212,0.031492,1.087710,0.746981,0.690074,NaN,63.403712
3,0.146476,-1.167865,-0.111847,NaN,-0.800590,1.107721,0.644311,-0.328375,0.566602,NaN,95.251323
4,-2.205566,-0.635362,-1.876553,0.619711,1.274875,-0.624345,NaN,-1.189667,NaN,NaN,-299.921204
...,...,...,...,...,...,...,...,...,...,...,...
995,0.428186,-0.596974,0.379153,1.230222,-1.279031,-0.412221,NaN,-2.390304,0.913474,-0.279993,28.140329
996,1.647878,0.562854,-1.088693,0.561473,0.126313,0.780202,-0.355935,-0.137741,-1.174202,0.068322,129.316401
997,-1.506534,2.492184,0.249451,0.399228,-0.587115,-0.317761,NaN,-0.275121,0.116579,-1.965556,-50.840705
998,NaN,1.194109,-1.725807,-0.677565,NaN,-0.464404,0.783391,-0.981166,NaN,-0.597510,-102.684755


In [45]:
tf1 = ColumnTransformer(
    [
     ("fill_missing" ,SimpleImputer(strategy = "mean") , slice(0,10) )
    ]
)

In [54]:
tf2  = ColumnTransformer(
    transformers=[
        ("scaling" , MinMaxScaler() , slice(0,10))
    ]
)

In [51]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
tf3= ColumnTransformer(
    transformers = [
        ("selection",RFE(estimator = LinearRegression() , n_features_to_select = 6) , slice(0,10))
    ]
)

In [57]:
from sklearn.linear_model import SGDRegressor
tf4 = SGDRegressor(max_iter = 1000  , tol = 1e-3 , eta0 = 0.01)

In [58]:
pipe = Pipeline(
    [
        ("tf1" , tf1),
        ("tf2" , tf2),
        ("tf3" , tf3),
        ("tf4"  , tf4)
    ]
)

In [62]:
pipe.fit(x_train , y_train)

Pipeline(steps=[('tf1',
                 ColumnTransformer(transformers=[('fill_missing',
                                                  SimpleImputer(),
                                                  slice(0, 10, None))])),
                ('tf2',
                 ColumnTransformer(transformers=[('scaling', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('tf3',
                 ColumnTransformer(transformers=[('selection',
                                                  RFE(estimator=LinearRegression(),
                                                      n_features_to_select=6),
                                                  slice(0, 10, None))])),
                ('tf4', SGDRegressor())])

In [65]:
y = pipe.predict(x_test)

In [68]:
from sklearn.metrics import r2_score
r2_score(y_test , y)

0.9532108357128769

In [70]:
from sklearn.model_selection import cross_val_score

In [73]:
np.mean(cross_val_score(pipe , x_train, y_train , scoring ="r2" ,cv=5))

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_stochastic_gradient.py:1608: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_stochastic_gradient.py:1608: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


np.float64(0.8880436862755431)